# Lab 10: Object Detection with YOLO

**Duration:** 3 hours. **Level:** beginner. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
A classifier says *what* is in the picture. A **detector** says *what and where*: it finds every object it knows, draws a **box** around it and gives its class and a score. This is the most used task of computer vision, and the base for tracking, counting and alarms in the next labs. In this lab you run a pretrained YOLO detector on pictures and on live video, read the numbers of a box, learn what the confidence limit does, and measure the speed of the model on your Jetson.

**By the end of this lab you can:**
1. Explain the difference between classification and detection.
2. Run a pretrained YOLO detector on pictures and on live video.
3. Read the output of a box: class, confidence and the corners `x1, y1, x2, y2`.
4. Explain what the confidence limit does and find a good value for a job.
5. Measure the model-only speed of two YOLO sizes, and record the baseline for Lab 15.

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. Pictures and live video appear under the cell that makes them. Press **Ctrl + S** often.

> **No training in this lab.** The model was trained by Ultralytics on the COCO collection (80 object classes). You only use it.

## 2. Background

**Detection.** A detector finds **many objects** in one picture. For every object it gives three things: a **class** (for example `person`), a **confidence** (a score between 0 and 1) and a **box** (four numbers).

**The box.** The box is written as `x1, y1, x2, y2`: the top-left corner and the bottom-right corner, in pixels. The point (0, 0) is the top-left corner of the picture. So the width of the box is `x2 - x1` and the height is `y2 - y1`.

**COCO, 80 classes.** The model knows 80 everyday classes: person, bicycle, car, bus, dog, cup, laptop, phone and more. It can **not** find an object that is not in the list. It does not say *unknown*. It simply draws no box (or a wrong box).

**The confidence limit.** The model proposes many boxes. Only the boxes with a confidence **above the limit** are shown.

| Limit | Effect |
|---|---|
| Low (0.10) | More objects found, but also more wrong boxes (false alarms) |
| High (0.80) | Only very sure boxes, but real objects can be missed |

There is no best value. A safety job wants a low limit (do not miss a person). An alarm that sends an SMS wants a higher limit (do not wake people for nothing).

**Model sizes.** YOLO26 comes in sizes: `n` (nano), `s` (small), and bigger ones. A bigger model is usually more accurate and slower. On the Jetson `n` and `s` are the useful ones.

| Name | Size | Use |
|---|---|---|
| `yolo26n.pt` | smallest and fastest | live video, first choice |
| `yolo26s.pt` | a little bigger | when `n` misses objects |

**Latency and FPS.** Latency is the time for one picture in milliseconds. FPS = 1000 / latency (ms). In Part E you measure both.

**Change one thing at a time.** If you break the notebook, close it and run `import labview; labview.reset(10)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: Prepare (15 min)

### Step 1: A Helper to Show Pictures

In [ ]:
from IPython.display import Image, display

def show(*names, width=360):
    """Show picture files under the cell."""
    for n in names:
        print(n)
        display(Image(filename=n, width=width))

### Step 2: Your Workspace
This notebook lives in the `lab10` folder inside your own folder. Check where you are:

In [ ]:
%%bash
pwd
ls

### Step 3: Check PyTorch and YOLO
If `CUDA available` is `False` or the PyTorch version ends with `+cpu`, stop and call your instructor. Do **not** run `pip install` in this lab.

In [ ]:
import torch
import ultralytics

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
print("Ultralytics version:", ultralytics.__version__)

### Step 4: Get the Sample Pictures and Models
The YOLO package comes with two sample pictures. This copies them into the folder of this notebook. It also copies the model files that the Lab 0 setup downloaded.

In [ ]:
import glob, os, shutil
import ultralytics

assets = os.path.join(os.path.dirname(ultralytics.__file__), "assets")
for f in glob.glob(assets + "/*.jpg"):
    shutil.copy(f, ".")

# Model files downloaded by the Lab 0 setup
models = os.path.join(os.environ["LAB_HOME"], "models")
for f in glob.glob(models + "/*.pt"):
    shutil.copy(f, ".")

print(sorted(glob.glob("*.jpg")))
print(sorted(glob.glob("*.pt")))
show("bus.jpg", "zidane.jpg")

---
# Part B: Detect Objects in Pictures (45 min)

### Step 1: Detect Objects
The detector finds every object it knows, draws a box around it and gives its class and confidence. The second value, `conf`, is the confidence limit.

In [ ]:
from ultralytics import YOLO
import torch

device = 0 if torch.cuda.is_available() else "cpu"
import cv2

det_model = YOLO("yolo26n.pt")

def detect(image, conf=0.25, save_as="detected.jpg"):
    result = det_model(image, conf=conf, device=device, verbose=False)[0]
    print(f"Image: {image}   Confidence limit: {conf}")
    print(f"Found {len(result.boxes)} objects:")
    for box in result.boxes:
        name = result.names[int(box.cls)]
        x1, y1, x2, y2 = [int(v) for v in box.xyxy[0]]
        print(f"  {name:<12} {float(box.conf):.2f}   box: ({x1}, {y1}) to ({x2}, {y2})")
    cv2.imwrite(save_as, result.plot())
    return result

detect("bus.jpg")
show("detected.jpg")

In [ ]:
detect("zidane.jpg")
show("detected.jpg")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- How many objects were found in bus.jpg? Which classes?:
- How many in zidane.jpg? Is there an object the model missed?:
- Which object had the lowest confidence, and why do you think so?:

### Step 2: Change the Confidence Limit
This runs the detector on `bus.jpg` with four different limits. It saves one picture for each, so you can compare the lowest and the highest.

In [ ]:
for limit in [0.10, 0.25, 0.50, 0.80]:
    detect("bus.jpg", conf=limit, save_as=f"detected_{limit}.jpg")
    print()

show("detected_0.1.jpg", "detected_0.8.jpg")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Number of objects found at limit 0.10, 0.25, 0.50 and 0.80::
- What wrong boxes (false alarms) did you see at 0.10?:
- Which real object disappeared first when the limit went up?:

### Step 3: Look at the Box Numbers
Work out the width and height of the **bus** box by hand first (`x2 - x1` and `y2 - y1`). Then run this cell to check your answer.

In [ ]:
result = det_model("bus.jpg", conf=0.25, device=device, verbose=False)[0]
for box in result.boxes:
    if result.names[int(box.cls)] == "bus":
        x1, y1, x2, y2 = [int(v) for v in box.xyxy[0]]
        print("Bus box:", (x1, y1), "to", (x2, y2))
        print("Width :", x2 - x1)
        print("Height:", y2 - y1)
        print("Touches the left edge:", x1 <= 2)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Width and height of the bus box (your calculation)::
- Does the box touch the edge of the picture? What does it mean for an object that is cut by the edge?:

### Step 4: Classification or Detection?
Compare the Lab 9 result for `bus.jpg` with the detection result. Fill in the table (double-click to edit).

| | Lab 9 classifier | Lab 10 detector |
|---|---|---|
| Answer for bus.jpg | | |
| Does it say where? | | |
| Does it count objects? | | |

---
# Part C: Live Detection on the USB Camera (30 min)

Run the cell. The live picture with the boxes appears under it. Use the **Confidence** slider and watch the FPS. Press **Stop** when you are done, so the camera and the memory are free for Part E.

The FPS includes sending every picture to your browser. It is lower than the model-only speed you measure in Part E.

In [ ]:
import cv2
import labview

live_model = YOLO("yolo26n.pt")


def process(frame, state):
    result = live_model(frame, conf=state["Confidence"], device=device, verbose=False)[0]
    shown = result.plot()
    counts = {}
    for c in result.boxes.cls:
        name = result.names[int(c)]
        counts[name] = counts.get(name, 0) + 1
    summary = ", ".join(f"{n}: {k}" for n, k in counts.items()) or "nothing found"
    cv2.putText(shown, f"{state['fps']:.0f} FPS  conf {state['Confidence']:.2f}", (10, 28),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    cv2.putText(shown, summary, (10, 58), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    return shown


live = labview.live(process, controls={"Confidence": (0.05, 0.95, 0.05, 0.40)})

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- FPS at confidence 0.40::
- Which objects did the model find easily?:
- What did the slider do when you moved it down to 0.05 and up to 0.90?:

### Step 2: Find the Weak Spots
Run the live cell again and try to fool the model. Turn the camera so an object is only half visible. Hold up an object that is not one of the 80 classes (for example a pen cap or a stapler). Move an object quickly. Lower the light by covering the lamp with a book. Write down what you find (double-click to edit):

| Test | What happened |
|---|---|
| Object half out of view | |
| Object not in the 80 classes | |
| Fast movement | |
| Dim light | |

> **Note.** The FPS shown includes everything: reading the camera, running the model, drawing the boxes and sending the picture to your browser.

---
# Part D: Pick a Confidence Limit for a Job (20 min)

Choose **one** job and pick the limit you would use. Test it on the live camera for two minutes, then explain your choice.

| Job | Which mistake is worse? |
|---|---|
| A: Count the cups on a shelf for a report | |
| B: Warn when a person enters a danger zone | |
| C: Switch on a light when someone is in the room | |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Job you chose::
- Limit you picked and why::
- What mistake did you see during the two minutes (false alarm or missed object)?:

---
# Part E: Measure the Speed Baseline (30 min)

### Step 1: Check the Power Mode
This measures only the time the model needs for one picture, for the nano and the small model. It runs 5 warm-up passes and then 30 timed runs. Close the live window and your other notebooks first. Check the power mode first with the two cells below (the first asks for your password for `sudo`), and write it in your table.

In [ ]:
import labview
labview.sudo_login()

In [ ]:
!sudo nvpmodel -q

### Step 2: Run the Benchmark
Run the cell **twice** and record the second run. You need these numbers in Lab 15 (TensorRT).

In [ ]:
import time
import torch
from ultralytics import YOLO

image = "bus.jpg"
device = 0 if torch.cuda.is_available() else "cpu"
names = ["yolo26n.pt", "yolo26s.pt"]
runs = 30

for name in names:
    model = YOLO(name)
    for _ in range(5):                       # warm-up
        model(image, imgsz=640, device=device, verbose=False)

    times = []
    for _ in range(runs):
        r = model(image, imgsz=640, device=device, verbose=False)[0]
        times.append(r.speed["inference"])   # milliseconds, model only

    avg = sum(times) / len(times)
    print(f"{name:<12} average inference: {avg:6.1f} ms   ({1000 / avg:5.1f} FPS)")

| Model | Average inference (ms) | FPS | Power mode |
|---|---|---|---|
| yolo26n.pt | | | |
| yolo26s.pt | | | |

### Try It Yourself
- Change `imgsz=640` to `320` in the cell above. How does the time change? Does the model still find the small objects?

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Average inference time of yolo26n and yolo26s::
- How many times slower is the small model?:
- Why is the live FPS lower than the model-only FPS?:

> **Where next.** In the next lab you use a detector that gives **rotated** boxes (oriented bounding boxes). Later labs use detection for tracking, counting and alarms, and Lab 15 makes this speed faster with TensorRT.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | The detector runs on `bus.jpg` and `zidane.jpg` and the boxes are shown | |
| 2 | The confidence experiment (four limits) is done and you explain it | |
| 3 | You calculate the width and height of a box and the answer matches the cell | |
| 4 | The live detection runs with boxes and the FPS on the screen | |
| 5 | The weak-spot table and the confidence-limit job are filled in | |
| 6 | The speed baseline table (nano and small) is complete, with the power mode | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers**

- 1. What is the difference between classification and detection? Give one job for each.:
- 2. A box is `(100, 50)` to `(300, 250)`. What are the width and the height?:
- 3. What does the confidence limit do? What happens if you set it to 0.95?:
- 4. Why does the detector miss an object that is not in the 80 classes, and not say `unknown`?:
- 5. A box touches the edge of the picture. What does this mean?:
- 6. Which is better for a safety camera: a low or a high limit? Explain.:
- 7. The inference time is 20 ms. What is the FPS? Why is the live FPS lower?:
- 8. Why must you close other notebooks before the benchmark?:
- 9. Name two things that can make a detector fail in a real shop.:

## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| `CUDA available: False` | The PyTorch of your environment is the CPU version | Stop and call your instructor. Do not run `pip install`. |
| `yolo26n.pt` not found | The Lab 0 setup did not download the model files | Ask your teacher. |
| `Cannot open the camera` | Another program uses the camera | Press Stop in other live cells, shut down other kernels, ask your teacher to check the camera on the teacher page |
| No boxes appear | The confidence limit is too high, or the object is not in the 80 classes | Lower the limit. Try a person or a cup. |
| The benchmark is much slower than your classmates | Another program uses the GPU, or the power mode is low | Close other notebooks. Check `nvpmodel` in Step 1 of Part E. |
| `CUDA out of memory` | Other notebooks use the shared 8 GB | Press Stop, shut down other kernels (Kernel, Shut Down All Kernels) and run again |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(10)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- Ultralytics, Object Detection: https://docs.ultralytics.com/tasks/detect/
- COCO dataset (the 80 classes): https://cocodataset.org/